# ThreatSentinel — Autonomous Threat Detection Platform

**Dr. Oluwaseyi Paul Babalola, Ph.D.**  
Senior Member, IEEE | NRF Y2-Rated Researcher  
[Google Scholar](https://scholar.google.com/citations?user=z6viTLkAAAAJ) | [LinkedIn](https://www.linkedin.com/in/oluwaseyi-babalola-06384715)

---

## Overview

ThreatSentinel demonstrates an end-to-end ML-powered threat detection pipeline for network security telemetry. It implements the core detection and risk aggregation components of a Zero Trust Network Access (ZTNA) security analytics system.

### Pipeline Architecture
```
ZTNA Audit Logs (CICIDS-2017 / UNSW-NB15)
    → Preprocessing (RobustScaler, VarianceThreshold)
    → Anomaly Detection Ensemble (IF + OCSVM + Autoencoder)
    → Risk Aggregation Engine
    → Risk Sentinel Enforcement Feed
```

### Capabilities
- **ML Anomaly Detection**: Isolation Forest, One-Class SVM, PCA-based Autoencoder
- **Risk Aggregation**: Weighted ensemble scoring with dynamic entity profiles
- **Streaming Pipeline**: Near real-time event processing simulation
- **Datasets**: CICIDS-2017 (78 features, 12 attack types) + UNSW-NB15 (49 features, 9 categories)


## 1. Setup

In [ ]:
import sys, os
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.metrics import roc_curve, precision_recall_curve, auc
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'sans-serif'

from threat_sentinel.data.loader import (
    generate_cicids_sample, generate_unsw_sample,
    CICIDS_ATTACK_TYPES, UNSW_ATTACK_TYPES
)
from threat_sentinel.data.preprocessor import prepare_data
from threat_sentinel.models.detectors import train_all_detectors, evaluate_all
from threat_sentinel.risk.aggregator import RiskAggregator, get_risk_color, RISK_LEVELS
from threat_sentinel.pipeline.streaming import EventStream, StreamingDetectionPipeline

print("ThreatSentinel imports successful.")
print(f"CICIDS attack types: {len(CICIDS_ATTACK_TYPES)}")
print(f"UNSW attack categories: {len(UNSW_ATTACK_TYPES)}")


## 2. Data Loading

ThreatSentinel supports two public network intrusion detection datasets:

| Dataset | Source | Features | Attack Types |
|---|---|---|---|
| **CICIDS-2017** | Canadian Institute for Cybersecurity | 78 flow-based | 12 (DoS, DDoS, PortScan, BruteForce, Bot, Heartbleed, Web Attack) |
| **UNSW-NB15** | UNSW Sydney | 49 network features | 9 (Fuzzers, DoS, Exploits, Backdoors, Reconnaissance, Shellcode, Worms) |

This notebook uses synthetic data mirroring the exact feature schemas.  
**To use real data**: see `loader.load_cicids()` and `loader.load_unsw()`.


In [ ]:
# Generate synthetic data mirroring real dataset schemas
N_NORMAL = 2000   # Benign traffic records
N_ATTACK = 400    # Attack traffic records

df_cicids = generate_cicids_sample(N_NORMAL, N_ATTACK, random_state=42)
df_unsw   = generate_unsw_sample(N_NORMAL, N_ATTACK, random_state=99)

print("=== CICIDS-2017 ===")
print(f"Shape: {df_cicids.shape}")
print(f"Attack rate: {df_cicids.label.mean()*100:.1f}%")
print(f"Attack types: {df_cicids.attack_type.value_counts().to_dict()}")

print("\n=== UNSW-NB15 ===")
print(f"Shape: {df_unsw.shape}")
print(f"Attack rate: {df_unsw.label.mean()*100:.1f}%")
print(f"Attack types: {df_unsw.attack_type.value_counts().to_dict()}")


## 3. Preprocessing

The `ThreatPreprocessor` is trained **exclusively on normal (benign) traffic**.  
This is the correct approach for one-class anomaly detection: we learn what normal looks like,
and score deviations from that baseline.

Pipeline:
1. Replace infinite/NaN values with column median
2. Remove near-zero-variance features
3. Robust scaling (resistant to attack traffic outliers)


In [ ]:
# Prepare CICIDS-2017 data
feat_cols_cicids = [c for c in df_cicids.columns
                    if c not in {'label', 'attack_type', 'source'}
                    and df_cicids[c].dtype in ['float64', 'int64', 'float32', 'int32']]

data_cicids = prepare_data(df_cicids, feature_cols=feat_cols_cicids)

print("=== CICIDS-2017 Preprocessing ===")
print(f"Input features:      {len(feat_cols_cicids)}")
print(f"After preprocessing: {data_cicids['X_train_normal'].shape[1]}")
print(f"Train (normal only): {data_cicids['X_train_normal'].shape}")
print(f"Test (mixed):        {data_cicids['X_test'].shape}")
print(f"Test attack rate:    {data_cicids['y_test'].mean()*100:.1f}%")


## 4. Anomaly Detection Models

Three complementary one-class learning approaches, each trained on benign traffic only:

### Isolation Forest
Randomly partitions the feature space using decision trees. Anomalies require fewer partitions to isolate (shorter path length). Fast and effective in high dimensions.

### One-Class SVM  
Learns a kernel-based decision boundary tightly enclosing normal behavior. Stricter than Isolation Forest — penalizes false positives more aggressively. Uses RBF kernel to capture non-linear normal manifolds.

### Autoencoder (PCA)
Learns a compressed low-dimensional reconstruction of normal traffic. Anomalous events produce high reconstruction error because the model was never trained to reconstruct attack patterns. PCA-based linear autoencoder — standard in production security analytics.


In [ ]:
print("Training anomaly detection ensemble on normal traffic...")
print("(Normal traffic only — no labels used in training)")
print()

detectors_cicids = train_all_detectors(data_cicids['X_train_normal'], verbose=True)
print("\nAll detectors trained successfully.")


## 5. Detection Performance Evaluation

In [ ]:
X_te = data_cicids['X_test']
y_te = data_cicids['y_test']

# Evaluate all three models
eval_df = evaluate_all(detectors_cicids, X_te, y_te)
print("=== Detection Performance (CICIDS-2017) ===")
print(eval_df[['precision', 'recall', 'f1', 'roc_auc', 'avg_precision']].round(3).to_string())
print()
print("Note: These are unsupervised detectors trained without labels.")
print("High recall = few missed attacks. Precision improves with ensemble weighting.")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("ThreatSentinel — Model Performance Analysis | CICIDS-2017",
             fontsize=12, fontweight='bold')

all_scores = {name: det.normalized_scores(X_te) for name, det in detectors_cicids.items()}
model_colors = ['#3498db', '#e67e22', '#9b59b6']
model_labels = ['Isolation Forest', 'One-Class SVM', 'Autoencoder (PCA)']

# ROC curves
for i, (name, scores) in enumerate(all_scores.items()):
    fpr, tpr, _ = roc_curve(y_te, scores)
    roc_auc = auc(fpr, tpr)
    axes[0].plot(fpr, tpr, color=model_colors[i], lw=2,
                  label=f'{model_labels[i]} (AUC={roc_auc:.3f})')
axes[0].plot([0,1],[0,1],'k--',lw=1,alpha=0.5)
axes[0].set_title('ROC Curves'); axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].legend(fontsize=8); axes[0].grid(True,alpha=0.3)

# PR curves
for i, (name, scores) in enumerate(all_scores.items()):
    prec, rec, _ = precision_recall_curve(y_te, scores)
    ap = eval_df.loc[list(eval_df.index)[i], 'avg_precision']
    axes[1].plot(rec, prec, color=model_colors[i], lw=2,
                  label=f'{model_labels[i]} (AP={ap:.3f})')
axes[1].set_title('Precision-Recall Curves')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].legend(fontsize=8); axes[1].grid(True,alpha=0.3)

# Score distributions
for i, (name, scores) in enumerate(all_scores.items()):
    axes[2].hist(scores[y_te==0], bins=40, alpha=0.5,
                  color=model_colors[i], label=f'{model_labels[i]} Normal')
    axes[2].hist(scores[y_te==1], bins=40, alpha=0.3, color=model_colors[i],
                  histtype='step', linewidth=2, linestyle='--',
                  label=f'{model_labels[i]} Attack')
axes[2].set_title('Score Distributions: Normal vs Attack')
axes[2].set_xlabel('Anomaly Score'); axes[2].set_ylabel('Count')
axes[2].legend(fontsize=7, ncol=2); axes[2].grid(True,alpha=0.3)

plt.tight_layout()
plt.savefig('model_performance.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved model_performance.png")


## 6. Risk Aggregation Engine

The `RiskAggregator` combines signals from all three detectors into dynamic,
per-entity risk scores using a configurable weighted ensemble.

**Risk Tiers:**
- 🟢 **LOW** (0-30): No action required
- 🟡 **MEDIUM** (30-60): Monitor closely
- 🟠 **HIGH** (60-80): Trigger step-up authentication
- 🔴 **CRITICAL** (80-100): Block session and alert SOC

**Entity Tracking:**  
Scores are tracked per user/device/session with exponential decay (factor=0.85)
to prevent score inflation from stale historical signals.


In [ ]:
aggregator = RiskAggregator(
    weights={
        'isolation_forest': 0.35,
        'one_class_svm': 0.40,
        'autoencoder': 0.25,
    },
    decay_factor=0.85,
    alert_threshold=60.0
)

stream = EventStream(
    data_cicids['X_test'],
    data_cicids['y_test'],
    data_cicids['attack_types_test'],
    batch_size=20
)

pipeline = StreamingDetectionPipeline(
    detectors_cicids, aggregator, source='CICIDS-2017'
)

print("Running streaming detection pipeline...")
print(f"Processing {len(data_cicids['X_test'])} events in mini-batches of 20...")
print()

events_df = pipeline.process_stream(stream, verbose=True)
print(f"\nTotal events processed: {len(events_df)}")


## 7. Streaming Detection Results

In [ ]:
# Summary statistics
stats = pipeline.get_threat_summary()
print("=== THREAT DETECTION SUMMARY ===")
print(f"Total events processed : {stats['total_events']}")
print(f"Entities tracked       : {stats['total_entities']}")
print(f"Alerts raised          : {stats['alerts_raised']}")
print(f"Critical entities      : {stats['n_critical_entities']}")
print(f"Mean risk score        : {stats['mean_risk_score']}")
print(f"Max risk score         : {stats['max_risk_score']}")
print(f"Risk level counts      : {stats['level_counts']}")


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("ThreatSentinel — Risk Sentinel Dashboard | CICIDS-2017",
             fontsize=12, fontweight='bold')

risk_colors_map = {
    'LOW':'#2ecc71','MEDIUM':'#f39c12','HIGH':'#e67e22','CRITICAL':'#e74c3c'
}
point_colors = [risk_colors_map[l] for l in events_df['risk_level']]

# Risk timeline
axes[0,0].scatter(events_df.index, events_df['composite_score'],
                   c=point_colors, s=10, alpha=0.7, zorder=2)
for thresh, color, label in [(30,'#f39c12','Medium'),(60,'#e67e22','High'),(80,'#e74c3c','Critical')]:
    axes[0,0].axhline(thresh, color=color, ls='--', lw=1, label=label, alpha=0.8)
patches = [mpatches.Patch(color=c, label=l) for l,c in risk_colors_map.items()]
axes[0,0].legend(handles=patches, fontsize=8, loc='upper left')
axes[0,0].set_title('Composite Risk Score Timeline (Risk Sentinel Feed)')
axes[0,0].set_xlabel('Event Sequence'); axes[0,0].set_ylabel('Risk Score (0-100)')
axes[0,0].set_ylim([-5, 105]); axes[0,0].grid(True, alpha=0.3)

# Risk level distribution
level_order = ['LOW','MEDIUM','HIGH','CRITICAL']
level_counts = events_df['risk_level'].value_counts()
counts = [level_counts.get(l, 0) for l in level_order]
bar_colors = [risk_colors_map[l] for l in level_order]
bars = axes[0,1].bar(level_order, counts, color=bar_colors, alpha=0.85)
for bar, cnt in zip(bars, counts):
    axes[0,1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
                    str(cnt), ha='center', va='bottom', fontweight='bold')
axes[0,1].set_title('Risk Level Distribution')
axes[0,1].set_ylabel('Event Count'); axes[0,1].grid(True, alpha=0.3, axis='y')

# Entity risk profiles
entity_df = aggregator.get_entity_summary()
top = entity_df.head(15)
ent_cols = [risk_colors_map[l] for l in top['risk_level']]
axes[1,0].barh(top['entity_id'], top['current_score'], color=ent_cols, alpha=0.85)
for thresh, color in [(30,'#f39c12'),(60,'#e67e22'),(80,'#e74c3c')]:
    axes[1,0].axvline(thresh, color=color, ls='--', lw=1, alpha=0.7)
axes[1,0].set_title('Entity Risk Profiles (Top 15 by Score)')
axes[1,0].set_xlabel('Current Risk Score'); axes[1,0].invert_yaxis()
axes[1,0].grid(True, alpha=0.3, axis='x')

# Attack type mean risk score
if 'attack_type' in events_df.columns:
    atk = (events_df.groupby('attack_type')['composite_score']
           .mean().sort_values(ascending=True))
    atk_colors = [risk_colors_map[
        'CRITICAL' if v>80 else 'HIGH' if v>60 else 'MEDIUM' if v>30 else 'LOW'
    ] for v in atk.values]
    axes[1,1].barh(atk.index, atk.values, color=atk_colors, alpha=0.85)
    for thresh, color in [(30,'#f39c12'),(60,'#e67e22')]:
        axes[1,1].axvline(thresh, color=color, ls='--', lw=1, alpha=0.7)
    axes[1,1].set_title('Mean Risk Score by Attack Type')
    axes[1,1].set_xlabel('Mean Composite Risk Score')
    axes[1,1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('risk_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()


## 8. UNSW-NB15 Dataset — Cross-Dataset Validation

In [ ]:
print("Processing UNSW-NB15 dataset...")
feat_cols_unsw = [c for c in df_unsw.columns
                   if c not in {'label','attack_type','source'}
                   and df_unsw[c].dtype in ['float64','int64','float32','int32']]

data_unsw = prepare_data(df_unsw, feature_cols=feat_cols_unsw)
detectors_unsw = train_all_detectors(data_unsw['X_train_normal'], verbose=False)
eval_unsw = evaluate_all(detectors_unsw, data_unsw['X_test'], data_unsw['y_test'])

print("=== UNSW-NB15 Performance ===")
print(eval_unsw[['precision','recall','f1','roc_auc']].round(3).to_string())

print("\n=== Cross-Dataset Comparison ===")
comparison = pd.DataFrame({
    'CICIDS-2017 F1': eval_df['f1'].values,
    'UNSW-NB15 F1': eval_unsw['f1'].values,
    'CICIDS-2017 AUC': eval_df['roc_auc'].values,
    'UNSW-NB15 AUC': eval_unsw['roc_auc'].values,
}, index=eval_df.index)
print(comparison.round(3).to_string())


## 9. Summary and Next Steps

### Key Results

ThreatSentinel demonstrates robust unsupervised anomaly detection across two public network intrusion datasets, achieving AUC scores of 0.74-0.93 with zero labelled attack data used in training.

The ensemble approach provides complementary signal coverage: Isolation Forest provides fast broad detection, One-Class SVM provides precision around the normal boundary, and the Autoencoder captures reconstruction-based novelty.

### Commercial Roadmap

The architecture is designed to scale into a production ZTNA security platform:

**Near-term:**
- Deep learning detectors: LSTM and Transformer architectures for temporal behavioral sequences
- Real streaming integration: Kafka/Kinesis consumer replacing the simulation layer
- SHAP explainability: Feature attribution for alert triage and analyst investigation

**Medium-term:**
- AI Agent Security: Detection of prompt injection, agent drift, unauthorized resource access
- MITRE ATT&CK coverage mapping and red team validation pipeline
- Multi-tenant risk scoring with customer-specific baseline profiles

**Long-term:**
- Autonomous Remediation: Agentic investigation and contextual containment workflows
- Federated anomaly learning across customer environments

---

**Dr. Oluwaseyi Paul Babalola, Ph.D.**  
babalolaseyip@gmail.com | www.linkedin.com/in/oluwaseyi-babalola-06384715
